In [1]:
### Hayden Gallo
### 7/23/26
### Bucci Lab

### Here we are plotting the flux sampling simulations for the static timepoint Venturelli data from Clark et al. 2021

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import scipy.stats
import glob
import itertools
from tqdm import tqdm


In [2]:
import socket
print(socket.gethostname())

hpcc04


In [3]:
### print working dir

print(os.getcwd())

/home/hayden.gallo-umw/DySCoMeMo_master_branch/glv_dfba


In [4]:
### here set the flux sampling directory

flux_samp_base_dir = '/home/hayden.gallo-umw/DySCoMeMo_flux_sampling_sims/Venturelli_in_vitro/panGenusmodel_sims/test_12'

In [5]:
MET_COLS = ["EX_but(e)", "EX_ac(e)", "EX_lac_L(e)", "EX_succ(e)"]
MET_NAMES = {  # friendly names for output/plots
    "EX_but(e)": "Butyrate",
    "EX_ac(e)": "Acetate",
    "EX_lac_L(e)": "Lactate",
    "EX_succ(e)": "Succinate",
}

In [6]:
def load_flux_sampling_results(base_dir, met_cols=MET_COLS):
    """
    Walk base_dir/{experiment}/result_sim_*.pkl and build a long DataFrame:
    columns = ['experiment', 'draw', <met_cols...>]
    """
    rows = []
    exp_dirs = sorted(
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d))
    )
 
    for exp_name in tqdm(exp_dirs):

        if exp_name == 'params.txt' or exp_name == 'job_output' or exp_name == 'lsf_scripts':
        
            continue

        else:
            exp_dir = os.path.join(base_dir, exp_name)
            #print(exp_dir)
            pkl_paths = sorted(glob.glob(os.path.join(exp_dir, "sim_*.pkl")))
    
            for pkl_path in pkl_paths:
                #print(pkl_path.split('/')[-1].split('_')[1])
                draw_idx = int(pkl_path.split('/')[-1].split('_')[1])
                with open(pkl_path, "rb") as f:
                    result = pickle.load(f)
    
                met_pred = result["met_predictions"]  # Series (single timepoint) or DataFrame (multi)
                row = {"experiment": exp_name, "draw": draw_idx}
    
                if isinstance(met_pred, pd.Series):
                    for bigg_id in met_cols:
                        row[bigg_id] = met_pred.get(bigg_id, np.nan)
                else:
                    # multi-timepoint: take final available timepoint
                    last_row = met_pred.iloc[-1]
                    for bigg_id in met_cols:
                        row[bigg_id] = last_row.get(bigg_id, np.nan)
    
                rows.append(row)
    
        if not rows:
            raise FileNotFoundError(f"No result_sim_*.pkl files found under {base_dir}")
 
    return pd.DataFrame(rows)

In [7]:
### here build dataframe of flux sampling sims

flux_samp_df = load_flux_sampling_results(flux_samp_base_dir)

100%|██████████| 4/4 [00:00<00:00, 17.08it/s]


In [8]:
### here let's save the df so we don't have to reload every time 
flux_samp_df_file_name = flux_samp_base_dir + '/flux_samp_df.csv'
flux_samp_df.to_csv(flux_samp_df_file_name)
